# 한계효용 이론 검증

**검증할 가설**

> 다음 정보 1단위의 한계효용(MU = p(1−p)의 기대 감소)이 비용보다 클 때만 MU가 가장 큰 것부터 산다.
> 그러면 총효용은 따로 관리하지 않아도 (a) 플랜 A보다 항상 크고, (b) 시간이 갈수록 커진다.

**실험 구성** (프레임워크의 `extraction` / `online` 구조를 최소한으로 재현)

- 사건 1,500개 (실제 경기 1,517개와 같은 규모). 사건마다 결과 y ∈ {0, 1}.
- **플랜 A**: 정보를 사지 않는 기본 모델. 참 확률에 잡음이 섞인 확률 p0.
- **채널 40개**: 각 채널은 정확도 a(0.48~0.64)로 y를 맞히는 신호를 냄. 하나 살 때마다 비용 0.003.
- **파이프라인**: p0에서 시작해 MU가 가장 큰 채널을 MU > 비용인 동안 하나씩 사고 베이즈 갱신.
  결과가 확정되면 사용한 채널의 정확도 기록을 갱신.
- **측정**: 사건마다 Δ = Brier(플랜 A) − [Brier(파이프라인) + 비용]. Δ > 0이면 파이프라인이 이득.
  **예상 이득**(구매 시점에 계산한 MU − 비용의 합)도 같이 기록해, 실현 이득과 비교함.

**시나리오**: 앞 대화에서 짚은 전제 위반 조건(승자의 저주, 베낀 정보, 환경 변화)을 하나씩 켬.

## 1. 시뮬레이터

In [ ]:
import numpy as np

def logit(p): return np.log(p / (1 - p))
def sig(x): return 1 / (1 + np.exp(-x))

def exp_pq_after(p, a):
    """정확도 a인 신호 하나를 본 뒤의 기대 p'(1-p') (신호 독립 가정)."""
    ps1 = p * a + (1 - p) * (1 - a)
    p1 = p * a / ps1
    p0 = p * (1 - a) / (1 - ps1)
    return ps1 * p1 * (1 - p1) + (1 - ps1) * p0 * (1 - p0)

def run(seed, estimator="naive", copy=False, drift=False,
        T=1500, K=40, cost=0.003, n0=10, planA_noise=0.4,
        prior_n=20, z=1.0, forget=1.0,
        cusum_h=None, fast=0.95, explore=0.0,
        event_delay=None, event_recall=1.0, false_events=0):
    """
    사건 T개를 순서대로 처리. 사건마다 플랜 A 확률 p0에서 시작해
    한계효용(MU = p(1-p)의 기대 감소)이 가장 큰 채널을 MU > 비용인 동안 하나씩 산다.
    반환: 사건별 실현 이득 Δ = Brier(플랜A) - [Brier(파이프라인) + 비용],
          사건별 예상 이득 (구매 시점에 계산한 MU 합 - 비용)
    """
    rng = np.random.default_rng(seed)
    a_true = rng.uniform(0.48, 0.64, K)          # 채널의 실제 정확도
    src = np.arange(K)                           # 신호 출처 (베끼면 다른 채널)
    if copy:                                     # 절반 채널이 상위 5개를 베낌
        top = np.argsort(-a_true)[:5]
        for k in rng.choice(K, K // 2, replace=False):
            src[k] = rng.choice(top)
    acc = a_true.copy()                          # 원신호 정확도 (drift로 변함)
    # 외부 사건 기록 (휘슬): 별도 난수로 정해 다른 시나리오의 경로를 바꾸지 않음
    ev_rng = np.random.default_rng(seed + 10_000)
    events = {}                                  # 시점 -> 영향받는다고 기록된 채널
    if drift and event_delay is not None and ev_rng.random() < event_recall:
        events[T // 2 + event_delay] = "changed"  # 실제 변화를 알리는 사건
    for t_f in ev_rng.choice(T, false_events, replace=False):
        events.setdefault(int(t_f), ev_rng.choice(K, 8, replace=False))  # 헛휘슬: 무관한 채널 8개
    win = rng.binomial(n0, acc[src]).astype(float)  # 초기 이력 n0건
    n = np.full(K, float(n0))
    realized, predicted = np.zeros(T), np.zeros(T)
    wf, nf = win.copy(), n.copy()                # 짧은 기억 기록 (경보 때 교체용)
    cus = np.zeros(K)                            # 채널별 CUSUM

    for t in range(T):
        if drift and t == T // 2:                # 중간에 좋던 채널들이 무력화
            changed = (acc > 0.56)[src]
            acc = np.where(acc > 0.56, 0.47, acc)
        if t in events:                          # 외부 사건 기록 → 해당 채널 기록 삭제 (재검증 전까지 안 씀)
            hit_ch = changed if isinstance(events[t], str) else np.isin(np.arange(K), events[t])
            win[hit_ch], n[hit_ch], cus[hit_ch] = 0.5, 1.0, 0
        q = rng.beta(2, 2)
        y = bool(rng.random() < q)
        p0 = float(sig(logit(np.clip(q, .02, .98)) + rng.normal(0, planA_noise)))
        raw = np.where(rng.random(K) < acc, y, not y)
        s_k = raw[src]

        if estimator == "oracle":                # 참 정확도를 안다고 가정
            m, slack = acc[src], 0.0
        elif estimator == "shrink":              # 축소 + 불확실성만큼 깎기
            m = (win + 0.5 * prior_n) / (n + prior_n)
            slack = z * np.sqrt(m * (1 - m) / (n + prior_n))
        else:                                    # 관측 비율 그대로
            m, slack = win / n, 0.0
        strength = np.clip(np.abs(m - 0.5) - slack, 0, 0.45)
        a_use = 0.5 + strength                   # 신호를 쓸 때 가정하는 정확도
        flip = m < 0.5                           # 반대로 맞히는 채널은 뒤집어 읽음

        p, spent, gain_pred = p0, 0.0, 0.0
        used = np.zeros(K, bool)
        while True:
            mu = np.where(used, -1.0, p * (1 - p) - exp_pq_after(p, a_use))
            k = int(np.argmax(mu))
            if mu[k] <= cost:
                break
            used[k] = True
            spent += cost
            gain_pred += mu[k] - cost
            lr = np.log(a_use[k] / (1 - a_use[k]))
            p = float(np.clip(sig(logit(p) + (lr if (s_k[k] ^ flip[k]) else -lr)), 1e-4, 1 - 1e-4))

        checked = used.copy()                    # 검증만 하는 채널 (판단에는 안 씀)
        if explore and rng.random() < explore:
            idle = np.flatnonzero(~used)
            if idle.size:
                checked[rng.choice(idle)] = True; spent += cost
        realized[t] = (p0 - y) ** 2 - ((p - y) ** 2 + spent)
        predicted[t] = gain_pred
        win *= forget                            # 망각 계수 (1.0 = 안 잊음)
        n *= forget
        hit = (s_k == y)
        win[checked] += hit[checked]             # 결과 확정 후 사용·검증한 채널만 갱신
        n[checked] += 1
        wf[checked] = wf[checked] * fast + hit[checked]
        nf[checked] = nf[checked] * fast + 1
        if cusum_h is not None:                  # 경보 난 채널만 짧은 기억으로 교체
            ok = (hit ^ flip)                    # 뒤집어 읽은 기준으로 맞았는지
            ref = np.maximum(m, 1 - m)           # 기준: 채널 자신의 누적 적중률
            cus[used] = np.maximum(0, cus[used] + (ref[used] - ok[used]) - 0.05)
            bad = used & (cus > cusum_h)
            win[bad], n[bad], cus[bad] = wf[bad], nf[bad], 0
    return realized, predicted

def summarize(name, seeds=60, blocks=10, **kw):
    R, P = zip(*(run(s, **kw) for s in range(seeds)))
    R, P = np.array(R), np.array(P)
    T = R.shape[1]
    B = R.reshape(seeds, blocks, T // blocks).mean(2)
    first, second = R[:, :T // 2].mean(1), R[:, T // 2:].mean(1)
    return dict(
        시나리오=name,
        전체_실현이득=round(R.mean() * 1000, 2),
        예상이득=round(P.mean() * 1000, 2),
        과대평가비율=round(P.mean() / R.mean(), 2) if R.mean() > 0 else float("inf"),
        음수구간_비율=float(np.mean(B.mean(0) < 0)),
        후반이_전반보다_큰_시드=float(np.mean(second > first)),
        손해본_시드=float(np.mean(R.mean(1) < 0)),
        구간별=np.round(B.mean(0) * 1000, 2).tolist(),
    )

# 플랜 A: 공을 가졌을 때 3-2-5 (약한 축소로 MU 최대 채널을 공격적으로 구매),
#         경보가 나면 해당 채널은 4-4-2 (짧은 기억으로 교체) + 10% 검증 탐색
PLAN_A = dict(estimator="shrink", prior_n=5, z=0.3, cusum_h=8, explore=0.1)

SCENARIOS = {
    "1 이상적 (참 정확도 앎)":       dict(estimator="oracle"),
    "2 추정 (표본 10건)":            dict(estimator="naive"),
    "3 추정 + 베끼는 채널":          dict(estimator="naive", copy=True),
    "4 추정 + 중간 변화":            dict(estimator="naive", drift=True),
    "5 축소 추정":                   dict(estimator="shrink"),
    "6 축소 + 베끼는 채널":          dict(estimator="shrink", copy=True),
    "7 축소 + 중간 변화":            dict(estimator="shrink", drift=True),
    "8 약한 축소":                   dict(estimator="shrink", prior_n=5, z=0.3),
    "9 약한 축소 + 중간 변화":       dict(estimator="shrink", prior_n=5, z=0.3, drift=True),
    "10 약한 축소 + 변화 + 망각":    dict(estimator="shrink", prior_n=5, z=0.3, drift=True, forget=0.99),
    "11 플랜 A (3-2-5 / 4-4-2)":     PLAN_A,
    "12 플랜 A + 중간 변화":         dict(PLAN_A, drift=True),
}


## 2. 실행
시드 60개 기준, 코랩 CPU에서 몇 분 걸립니다. 빨리 보려면 `SEEDS = 20`으로 줄이세요.

In [ ]:
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

SEEDS = 60
with ProcessPoolExecutor() as ex:
    futs = {k: ex.submit(summarize, k, SEEDS, **v) for k, v in SCENARIOS.items()}
    results = [f.result() for f in futs.values()]

df = pd.DataFrame(results).set_index("시나리오")
df.drop(columns="구간별")

**열 읽는 법** (이득 단위: Brier × 1000, 사건 1개당)

| 열 | 뜻 | 가설이 맞다면 |
|---|---|---|
| 전체_실현이득 | 플랜 A 대비 실제 이득 평균 | > 0 |
| 예상이득 | 구매할 때 MU로 계산한 이득 | ≈ 실현이득 |
| 과대평가비율 | 예상 ÷ 실현. 1보다 크면 승자의 저주 | ≈ 1 |
| 음수구간_비율 | 시간 구간 10개 중 플랜 A보다 손해 본 구간 비율 | 0 |
| 후반이_전반보다_큰_시드 | 후반 750건의 이득이 전반보다 큰 시드 비율 | ≈ 1 ("시간이 갈수록 증가") |
| 손해본_시드 | 전체 이득이 음수인 시드 비율 | 0 |

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False
try:  # 코랩 한글 폰트 (없으면 영어 라벨로 대체)
    import subprocess
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
    import matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = "NanumGothic"
except Exception:
    pass

fig, ax = plt.subplots(figsize=(10, 5))
for r in results:
    ax.plot(range(1, 11), r["구간별"], marker="o", label=r["시나리오"])
ax.axhline(0, color="k", lw=1)
ax.set_xlabel("시간 구간 (사건 150개씩)")
ax.set_ylabel("플랜 A 대비 이득 (Brier ×1000)")
ax.set_title("시간에 따른 순이득: 0 아래면 플랜 A보다 손해")
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

## 3. 사전 실행 결과 (시드 60개)

| 시나리오 | 실현 이득 | 예상 이득 | 과대평가 | 손해 구간 | 후반 > 전반 | 손해 본 시드 |
|---|---|---|---|---|---|---|
| 1 이상적 (참 정확도 앎) | **+55.2** | 52.9 | 0.96 | 0% | 43% | 0% |
| 2 추정 (표본 10건) | +21.6 | 38.1 | **1.76** | 10% (첫 구간) | 85% | 2% |
| 3 추정 + 베끼는 채널 | +23.0 | 48.1 | **2.09** | 10% (첫 구간) | 98% | 2% |
| 4 추정 + 중간 변화 | **−8.3** | 28.6 | ∞ | **60%** | 3% | **100%** |
| 5 축소 추정 (강함) | +0.3 | 0.2 | 0.79 | 0% | 10% | 8% |
| 6 축소 + 베끼는 채널 | +0.5 | 0.5 | 1.05 | 0% | 15% | 15% |
| 7 축소 + 중간 변화 | +0.0 | 0.2 | 5.07 | 50% | 8% | 8% |
| 8 약한 축소 | +13.3 | 17.4 | 1.31 | 0% | 70% | 5% |
| 9 약한 축소 + 중간 변화 | −2.2 | 12.5 | ∞ | 50% | 5% | 95% |
| 10 약한 축소 + 변화 + 망각 | +1.6 | 4.3 | 2.63 | 10% | 42% | 42% |

**가설별 판정**

- **(a) "플랜 A보다 항상 크다"**: 정확도를 정확히 알 때(1번)만 성립합니다.
  추정치를 쓰면 초반에 손해를 봅니다(2, 3번 첫 구간 −13, −26).
  환경이 바뀌면 시드 100%가 전체적으로 손해입니다(4번).
- **(b) "시간이 갈수록 증가한다"**: 1번에서는 이득이 처음부터 일정하고 더 늘지 않습니다(후반 > 전반 43%, 우연 수준).
  2, 3번의 증가는 초반 손실이 회복되는 모습이지 계속 커지는 것이 아닙니다. 4번에서는 변화 후 계속 감소합니다.
- **승자의 저주**: MU 최댓값을 고르면 예상 이득이 실현 이득의 1.8~2.1배로 부풀려집니다.
  베끼는 채널이 있으면 더 커집니다(2.09).
- **보정의 대가**: 강한 축소는 과대평가는 없애지만 거의 사지 않아 이득도 0이 됩니다(5번). 약한 축소(8번)가 중간입니다.
- **망각**: 변화 후의 손실은 막지만, 모든 채널의 기록이 줄어들면 아무것도 사지 않는 상태에 갇힙니다(10번 후반 0.0).
  정리하신 원칙대로 망각은 **변화 감지(CUSUM)가 경보를 낼 때만** 적용하는 편이 맞아 보입니다.

**결론**: MU > 비용 규칙 자체는 맞습니다(1번). 하지만 총효용이 반드시 증가한다는 전제는
**추정 오차와 환경 변화가 있으면 깨집니다.** 그러니 총효용(Δ의 이동평균)은 최적화하지 않더라도 감시해야 합니다.

## 4. 직접 바꿔보기
`run()`의 인자를 바꿔 다른 조건을 시험할 수 있습니다.

- `cost`: 정보 1단위 비용. `n0`: 처음에 아는 채널별 이력 수 (작을수록 승자의 저주가 큼)
- `planA_noise`: 플랜 A의 잡음 (작을수록 플랜 A가 강함)
- `prior_n`, `z`: 축소 강도 (`estimator="shrink"`일 때)
- `forget`: 망각 계수 (1.0 = 안 잊음), `copy`, `drift`: 전제 위반 조건 켜기
- `cusum_h`: 4-4-2 전환 경보 기준 (None = 전환 없음), `explore`: 사건당 검증 탐색 확률, `fast`: 경보 때 쓰는 짧은 기억의 망각 계수
- `event_delay`: 외부 사건(휘슬)이 변화 뒤 몇 건 만에 기록되는지 (None = 휘슬 없음), `event_recall`: 변화가 기록될 확률, `false_events`: 헛휘슬 횟수

In [ ]:
r = summarize("내 실험", seeds=20, estimator="naive", n0=50, cost=0.003, planA_noise=0.4)
{k: v for k, v in r.items()}

## 5. 플랜 A: 공 있을 때 3-2-5, 공 없을 때 4-4-2

| 포지션 | 3-2-5 (공 있음 = 환경 안정, 정보 구매) | 4-4-2 (공 없음 = 경보, 손실 방어) |
|---|---|---|
| 최전방 | **5**: MU 상위 채널을 공격적으로 구매 (약한 축소 `prior_n=5, z=0.3`) | **2**: 검증 탐색 10% (`explore`) + LLM 원인 판단 |
| 미드필드 | **2**: MU 계산기 + 비용 규칙 (MU > 비용일 때만) | **4**: 검증 경로 4종 (결과 대조 · 원문 대조 · 재현 · 의견) |
| 수비 | **3**: 플랜 A 기본 모델 · as-of 기록 · 채널별 CUSUM | **4**: 위 3개 + 축소 추정기 복귀 |
| **인버티드 왼쪽 윙백** | **축소 추정기**가 미드필드로 들어가 MU 계산 옆에서 승자의 저주를 깎고, 역습(과대평가 손실) 1차 저지 | 수비 라인으로 복귀 |

- **전환 규칙(하나뿐)**: 채널별 CUSUM이 `cusum_h=8`을 넘으면 그 채널만 4-4-2로 전환해, 기록을 짧은 기억(`fast=0.95`)으로 교체합니다.
- 기준은 채널 자신의 누적 적중률입니다. 처음에는 추정 정확도(a_use)를 기준으로 썼는데, 승자의 저주로 부풀려진 값이라 안정 구간에서도 잘못된 경보가 계속 났습니다.

In [ ]:
cmp = {
    "공격만 (축소·전환 없음)":        dict(estimator="naive"),
    "공격만 + 변화":                  dict(estimator="naive", drift=True),
    "플랜 A":                         PLAN_A,
    "플랜 A + 변화":                  dict(PLAN_A, drift=True),
    "공격만, 검증 50건":              dict(estimator="naive", n0=50),
    "공격만, 검증 50건 + 변화":       dict(estimator="naive", n0=50, drift=True),
    "플랜 A, 검증 50건":              dict(PLAN_A, n0=50),
    "플랜 A, 검증 50건 + 변화":       dict(PLAN_A, n0=50, drift=True),
}
with ProcessPoolExecutor() as ex:
    futs = {k: ex.submit(summarize, k, SEEDS, **v) for k, v in cmp.items()}
    res2 = [f.result() for f in futs.values()]
pd.DataFrame(res2).set_index("시나리오").drop(columns="구간별")

### 사전 실행 결과 (시드 60개, 이득 = Brier ×1000, ± = 95% 구간)

| 구성 | 안정 | 중간 변화 | 변화 시 손해 본 시드 |
|---|---|---|---|
| 공격만, 초기 검증 10건 | **+21.6 ± 2.7** | −8.3 ± 0.6 | 100% |
| 플랜 A, 초기 검증 10건 | +10.8 ± 2.3 | **+3.6 ± 1.2** | 22% |
| 공격만, 초기 검증 50건 | **+35.9** | −11.2 | 100% |
| 플랜 A, 초기 검증 50건 | +23.9 | **+9.3** | **0%** |
| (참고) 공격만, 초기 검증 200건 (시드 40개) | +46.8 | −17.2 | 100% |
| (참고) 정확도를 정확히 앎 | +55.2 | | |

**읽는 법**
1. **한계효용 최대치를 가장 빨리 올리는 레버는 검증 깊이입니다.** 초기 검증 건수(`n0`)를 10 → 50 → 200으로 늘리면 이득이 21.6 → 35.9 → 46.8로 올라 상한 55.2에 다가갑니다.
   선택 규칙을 바꾸는 것보다 효과가 훨씬 큽니다.
2. **공격만 하는 구조는 변화가 생기면 모든 시드에서 손해입니다.** 검증을 많이 할수록 더 크게 집니다(−8 → −11 → −17). 과거 기록을 강하게 믿을수록 변화 뒤 손실이 커집니다.
3. **4-4-2 전환은 보험입니다.** 안정 구간에서 이득의 약 1/3을 내주는 대신, 변화가 생겨도 플랜 A보다 손해 보지 않게 합니다(검증 50건 기준 손해 본 시드 0%).
   변화가 드물면 공격만 하는 쪽이, 자주 있으면 플랜 A가 평균이 높습니다.
4. **남은 약점**
   - 변화 직후 한 구간(−15 ~ −30)은 막지 못합니다. 정답이 늦게 확정되는 만큼 변화 감지가 늦기 때문입니다.
   - 안정 구간 후반에 이득이 서서히 줄어듭니다. 잘못된 경보가 쌓이기 때문입니다.
   - 숫자 조정(`cusum_h`, `explore`)은 이 시뮬레이터 하나에서 결과를 보고 고른 값입니다. 실제 데이터로 다시 확인하기 전까지는 과적합일 수 있습니다.
   - 컴퓨터나 numpy 버전이 다르면 일부 시나리오가 소수점 아래에서 달라질 수 있습니다(±0.3 이내). 비교가 경계값 근처에서 갈리면 이후 경로가 달라지기 때문입니다. 결론은 바뀌지 않습니다.


## 6. 공 있을 때 / 없을 때를 가장 이익이 되게 구분하기

축구에서는 공을 뺏기는 순간 모두가 압니다. 이 시스템에서 "공 없음"은 **환경이 바뀐 상태**인데, 바뀌었는지는 결과가 쌓인 뒤에야 압니다.
그래서 구분의 가치는 세 가지로 정해집니다.

1. **완벽하게 구분할 때의 상한**: 변화 시점을 정확히 알면 얼마나 버는가
2. **경계 위치**: CUSUM 기준값(`cusum_h`). 높을수록 공을 오래 소유합니다. 잘못된 경보는 줄지만 대응이 늦어집니다.
3. **휘슬**: 감독 교체·부상·모델 버전 변경 같은 외부 사건 기록. 결과를 기다리지 않고 해당 채널을 바로 4-4-2로 보냅니다.
   - 시뮬레이터에서는 기록된 채널의 정확도 이력을 지우고, 재검증하기 전까지 쓰지 않습니다.
   - 헛휘슬(실제 변화와 무관한 사건)은 무관한 채널 8개의 이력을 지웁니다.

이 섹션은 설정이 많아 코랩 CPU에서 5~10분 걸립니다. 빨리 보려면 `SEEDS`를 20으로 줄이세요.


In [ ]:

def seed_stats(name, seeds, **kw):
    R = np.array([run(s, **kw)[0] for s in range(seeds)])
    T = R.shape[1]
    per = R.mean(1) * 1000
    return dict(시나리오=name, 이득=round(per.mean(), 1),
                구간95=round(1.96 * per.std(ddof=1) / np.sqrt(seeds), 1),
                변화후_이득=round(R[:, T // 2:].mean() * 1000, 1),
                손해본_시드=round(float(np.mean(per < 0)), 2))

def table(cfg):
    with ProcessPoolExecutor() as ex:
        futs = [ex.submit(seed_stats, k, SEEDS, **v) for k, v in cfg.items()]
        return pd.DataFrame([f.result() for f in futs]).set_index("시나리오")

# 6-1. 상한: 변화 시점을 아는 휘슬, 지연별 (공격만, 초기 검증 50건, 중간 변화)
A = dict(estimator="naive", n0=50, drift=True)
table({f"휘슬 지연 {d}": dict(A, event_delay=d) for d in [0, 10, 30, 100, 300]} | {"휘슬 없음": A})


In [ ]:

# 6-2. 경계 위치: 전환 기준과 검증 깊이
cfg = {}
for name, kw in {"공격만": dict(estimator="naive"),
                 "공격 + 기준 12": dict(estimator="naive", cusum_h=12),
                 "공격 + 기준 8": dict(estimator="naive", cusum_h=8),
                 "플랜 A": PLAN_A}.items():
    for n0 in [10, 50]:
        for d in [False, True]:
            cfg[f"{name} | 검증 {n0}건 | {'변화' if d else '안정'}"] = dict(kw, n0=n0, drift=d)
table(cfg)


In [ ]:

# 6-3. 휘슬의 가치와 헛휘슬의 비용 (공격 + 기준 12 + 탐색 5%, 초기 검증 50건)
B = dict(estimator="naive", n0=50, cusum_h=12, explore=0.05)
cfg = {}
for name, kw in {"기준 12 (휘슬 없음)": B,
                 "휘슬 지연 10 · 70% 기록": dict(B, event_delay=10, event_recall=0.7),
                 "휘슬 + 헛휘슬 3번": dict(B, event_delay=10, event_recall=0.7, false_events=3),
                 "휘슬 + 헛휘슬 10번": dict(B, event_delay=10, event_recall=0.7, false_events=10)}.items():
    for d in [False, True]:
        cfg[f"{name} | {'변화' if d else '안정'}"] = dict(kw, drift=d)
table(cfg)



### 사전 실행 결과 (시드 60개, 이득 = Brier ×1000, ± = 95% 구간)

**6-1. 완벽 구분의 상한** (공격만, 초기 검증 50건, 중간 변화)

| 휘슬 지연 | 전체 이득 | 변화 후 이득 | 손해 본 시드 |
|---|---|---|---|
| 0건 (즉시) | **+16.2 ± 1.5** | −0.6 | 0% |
| 10건 | +15.3 | −2.4 | 0% |
| 30건 | +13.7 | −5.6 | 0% |
| 100건 | +9.1 | −14.8 | 3% |
| 300건 | −0.8 | −34.5 | 60% |
| 휘슬 없음 | −11.2 | −55.3 | 100% |

- 공 없음을 즉시 알면 −11.2가 +16.2로 바뀝니다. 구분 자체의 가치는 약 27입니다.
- 즉시 전환해도 변화 후 이득은 0 근처입니다. **4-4-2의 목적은 득점이 아니라 실점 차단입니다.**
- 플랜 A(변화 시 +9.3)는 휘슬 지연 약 100건과 같은 수준입니다. CUSUM이 변화를 그만큼 늦게 잡습니다.

**6-2. 경계 위치**

| 구성 | 검증 10건 안정 | 검증 10건 변화 | 검증 50건 안정 | 검증 50건 변화 |
|---|---|---|---|---|
| 공격만 | **21.6** | −8.3 | **35.9** | −11.2 |
| 공격 + 기준 12 | 20.0 | +1.4 (손해 35%) | 33.9 | +4.2 (손해 20%) |
| 공격 + 기준 8 | 11.6 | +2.8 | 24.7 | +7.5 |
| 플랜 A (축소 + 기준 8 + 탐색 10%) | 10.8 | **+3.6** | 23.9 | **+9.3 (손해 0%)** |

- 경계를 하나로 정할 수는 없고, **안정 구간 이득과 변화 대응 사이의 교환**입니다. 변화가 드물면 기준 12, 잦으면 플랜 A가 낫습니다.
- **인버티드 윙백(축소 추정기)의 필요성은 검증 깊이에 달려 있습니다.**
  - 검증 50건이면 윙백을 올려 공격 + 기준 12로 가도 안정 구간 손실이 2뿐입니다.
  - 검증 10건이면 축소 추정이 없을 때 변화 구간에서 손해 본 시드가 35%로 늘어납니다.

**6-3. 휘슬의 가치와 헛휘슬의 비용** (공격 + 기준 12 + 탐색 5%, 초기 검증 50건)

| 구성 | 안정 | 변화 | 변화 시 손해 본 시드 |
|---|---|---|---|
| 기준 12 (휘슬 없음) | 35.1 | 3.4 | 27% |
| + 휘슬 (지연 10건, 변화의 70%만 기록) | **35.1** | **11.0 ± 1.6** | 3% |
| + 헛휘슬 3번 | 25.8 | 7.7 | 8% |
| + 헛휘슬 10번 | 11.5 | 3.1 | 38% |

- **진짜 휘슬은 거의 공짜입니다.** 안정 구간 이득은 그대로이고, 변화 구간 이득은 3.4에서 11.0으로 오릅니다.
  플랜 A(23.9 / 9.3)보다 안정과 변화 양쪽 모두 높습니다.
- **헛휘슬은 비쌉니다.** 한 번에 약 3씩 깎입니다. 이력을 지운 채널은 다시 검증할 때까지 쓰지 못하기 때문입니다.
  이 설정에서는 진짜 휘슬 1번의 가치(약 +7.6)가 헛휘슬 2~3번의 비용과 비슷합니다.
  **휘슬로 쓸 외부 사건은 "기록되면 실제로 바뀌는 사건"만 골라야 합니다.**

### 정리: 가장 이익이 되는 구분 방법
1. **공 있음 (3-2-5)**: 검증이 충분하면 윙백을 올리고 MU 최대 채널을 공격적으로 삽니다. 검증이 얕으면 윙백(축소 추정)을 안으로 좁힙니다.
2. **공 없음 (4-4-2) 전환 1순위: 휘슬.** 정밀한 외부 사건 기록으로 해당 채널을 바로 전환합니다.
3. **전환 2순위: CUSUM.** 기준을 높게(12) 두고, 휘슬이 놓친 변화만 잡는 백업으로 씁니다.
4. **탐색 5~10%**는 기록이 지워진 채널을 다시 쓸 수 있게 하는 최소 비용입니다.

**한계**
- 이 섹션의 설정은 같은 시뮬레이터에서 결과를 보고 고른 값입니다. 과적합일 수 있습니다.
- 헛휘슬의 비용은 "이력을 지운다"는 대응 방식에 달려 있습니다. 지우지 않고 짧은 기억으로 바꾸는 방식은 아직 시험하지 않았습니다.
- 실제 환경 변화 빈도와 실제 외부 사건의 정밀도는 실제 데이터에서 재야 합니다.



## 7. 실제 데이터로 검증: EPL 6,130경기 (2010/11 ~ 2026/27)

시뮬레이터 6번 섹션의 결론이 실제 데이터에서도 유지되는지 확인합니다.

**대응 관계**
- **예측 모델:** `llm-reliability/llmrel/epl.py`의 온라인 포아송 모델(골 모드 설정 그대로). 경기 전 정보만 써서 경기 순서대로 예측합니다.
- **플랜 A (정보 없음):** 직전 시즌까지의 홈승·무·원정승 비율.
- **채널:** 홈 이점 `h`. 이 채널의 기록을 어떻게 관리하는지만 바꿔 비교합니다.
- **환경 변화:**
  - **2020-06-17 무관중 재개:** 홈 이점이 사실상 사라졌습니다(홈 평균 골 차 +0.35 → +0.01).
  - **2021-08-13 관중 복귀:** 위 두 날짜는 실제로 기록된 외부 사건이라 **휘슬**로 씁니다.
  - **2024-25 시즌:** 원인이 알려지지 않은 홈 이점 감소(+0.09)가 있었습니다. CUSUM만으로 잡아야 하는 변화입니다.
- **이득:** 플랜 A 대비 RPS(순위 확률 점수, 축구 예측 표준 손실) 감소 × 1000.

**과적합을 막기 위해 미리 정한 것**
- **CUSUM 허용 폭** `k = 0.1`: 잡고 싶은 변화 크기의 절반입니다. 홈 이점이 0으로 사라지는 변화가 약 0.2σ입니다.
- **CUSUM 기준** `14`(민감) / `16`(둔감): **무관중 이전 기간(2011~2019)의 잘못된 경보 수만 보고** 정했습니다.
  각각 시즌당 약 1번, 4시즌에 1번 경보가 나는 값입니다. 시뮬레이터의 기준 8과 12에 해당합니다.
- **경보·휘슬 대응:** 홈 이점 기록을 지우고 현재 추정값에서 20경기 무게로 새로 시작한 뒤, 다시 누적합니다.

**데이터 오류 수정:** 원본 `en1_2019-20.json`에서 2020년 7월 경기 66개가 2019년 7월로 적혀 있어, 읽을 때 바로잡습니다.

몇 분 걸립니다. 헛휘슬 실험을 20회 반복하기 때문입니다.


In [ ]:
import json, os, urllib.request
from math import exp, factorial
import numpy as np, pandas as pd

# ---- 7번 섹션: 실제 EPL 데이터 ----

SEASONS = [f"{y}-{str(y + 1)[2:]}" for y in range(2010, 2027)]
DATA_URLS = [
    "https://raw.githubusercontent.com/quanter02/chelsea-analytics/claude/great-dirac-3wwgzy/experiments/data_epl/en1_{s}.json",
    "https://raw.githubusercontent.com/openfootball/football.json/master/{s}/en.1.json",
]
LOCAL = "data_epl"   # 로컬에 있으면 사용, 없으면(코랩) 내려받음

def norm(t):
    t = t.strip()
    for suf in (" AFC", " FC"):
        if t.endswith(suf):
            t = t[:-len(suf)]
    return t[4:] if t.startswith("AFC ") else t

def load_season(s):
    f = os.path.join(LOCAL, f"en1_{s}.json")
    if os.path.exists(f):
        return json.load(open(f, encoding="utf-8"))["matches"]
    for u in DATA_URLS:
        try:
            return json.loads(urllib.request.urlopen(u.format(s=s), timeout=30).read())["matches"]
        except Exception:
            continue
    raise RuntimeError(f"{s} 시즌 자료를 받지 못했습니다")

def load():
    rows = []
    for s in SEASONS:
        for x in load_season(s):
            sc = x.get("score")
            ft = sc.get("ft") if isinstance(sc, dict) else (sc if isinstance(sc, list) and len(sc) == 2 else None)
            if not ft:
                continue
            date = x["date"]
            if s == "2019-20" and date < "2019-08-01":   # 원본 오류: 무관중 재개 후 2020년 7월 경기가 2019-07로 기록됨
                date = "2020" + date[4:]
            rows.append(dict(season=s, date=date, home=norm(x["team1"]), away=norm(x["team2"]), hg=ft[0], ag=ft[1]))
    df = pd.DataFrame(rows).sort_values("date", kind="stable").reset_index(drop=True)
    df["y"] = np.select([df.hg > df.ag, df.hg == df.ag], [0, 1], 2)
    return df

MAXG = 10
_F = np.array([factorial(i) for i in range(MAXG + 1)], float)
def probs(lh, la):
    g = np.arange(MAXG + 1)
    m = np.outer(np.exp(-lh) * lh ** g / _F, np.exp(-la) * la ** g / _F)
    m /= m.sum()
    return np.array([np.tril(m, -1).sum(), np.trace(m), np.triu(m, 1).sum()])

def rps(P, y):
    o = np.zeros(3); o[y] = 1
    return ((np.cumsum(P)[:2] - np.cumsum(o)[:2]) ** 2).sum() / 2

WHISTLES = ["2020-06-17", "2021-08-13"]   # 무관중 재개, 관중 복귀 (실제로 기록된 외부 사건)

def run_epl(df, h_mode="fixed", cusum_h=None, cusum_k=0.1, whistles=(), fast=0.98, n_prior=50, n_reset=20,
        h_fixed=0.25, k=0.04, c=0.8, p=-0.2, mu0=0.3):
    """
    팀 실력은 실제 코드(epl.py 골 모드)와 같은 온라인 포아송으로 갱신하고,
    홈 이점 h를 하나의 '채널'로 보고 그 기록을 어떻게 관리할지만 바꾼다.
      fixed : h=0.25 고정 (현재 실제 코드)
      long  : 전체 기록으로 추정 (검증 깊이 최대, 안 잊음)
      short : 항상 짧은 기억 (fast^경기수 가중)
    cusum_h : 홈 골 차 잔차의 양방향 CUSUM 기준 (표준화 단위), cusum_k: 허용 폭
              (k = 잡고 싶은 변화 크기의 절반. 홈 이점이 0으로 사라지는 변화 ≈ 0.2σ → k = 0.1)
    whistles: 외부 사건 날짜.
    경보·휘슬 → 홈 이점 기록을 지우고 현재 추정값에서 n_reset경기 무게로 새로 시작 (이후 다시 누적)
    """
    a, d = {}, {}
    mu = mu0
    season = None
    # 홈 이점 기록: 홈 골·원정 골의 가중 합 (사전값 n_prior경기만큼 h_fixed로)
    m0 = 1.4
    def fresh(hp, n=n_prior):
        return [n * m0 * exp(hp / 2), n * m0 * exp(-hp / 2)]
    rec = fresh(h_fixed)
    mode_short = h_mode == "short"
    cp = cm = 0.0
    wq = sorted(whistles)
    out = np.zeros(len(df)); hs = np.zeros(len(df)); alarms = []
    for i, r in enumerate(df.itertuples()):
        if r.season != season:
            teams = set(df.loc[df.season == r.season, "home"]) | set(df.loc[df.season == r.season, "away"])
            for t in list(a):
                a[t] *= c; d[t] *= c
            for t in teams - set(a):
                a[t], d[t] = p, p
            season = r.season
        while wq and r.date >= wq[0]:                       # 휘슬: 기록을 현재 추정에서 새로 시작
            h_now = np.log(rec[0] / rec[1])
            rec = fresh(h_now, n_reset); cp = cm = 0.0
            wq.pop(0)
        h = h_fixed if h_mode == "fixed" else float(np.log(rec[0] / rec[1]))
        hs[i] = h
        lh = exp(mu + h + a[r.home] - d[r.away]); la = exp(mu + a[r.away] - d[r.home])
        out[i] = rps(probs(lh, la), r.y)
        # 홈 이점 기록 갱신
        if mode_short:
            rec[0] *= fast; rec[1] *= fast
        rec[0] += r.hg * exp(-(a[r.home] - d[r.away])) ; rec[1] += r.ag * exp(-(a[r.away] - d[r.home]))
        # CUSUM: 홈 골 차 잔차 (실제 − 예상) / 표준편차
        if cusum_h is not None:
            z = ((r.hg - r.ag) - (lh - la)) / np.sqrt(lh + la)
            cp = max(0.0, cp + z - cusum_k); cm = max(0.0, cm - z - cusum_k)
            if cp > cusum_h or cm > cusum_h:
                alarms.append(r.date)
                rec = fresh(float(np.log(rec[0] / rec[1])), n_reset); cp = cm = 0.0
        # 팀 실력 갱신 (epl.py와 같은 방식)
        eh, ea = r.hg - lh, r.ag - la
        a[r.home] += k * eh; d[r.away] -= k * eh
        a[r.away] += k * ea; d[r.home] -= k * ea
        mu += k * 0.05 * (eh + ea)
    return out, hs, alarms

def plan_a(df):
    """플랜 A (정보 없음): 직전 시즌까지의 홈승·무·원정승 비율."""
    out = np.zeros(len(df))
    for s in SEASONS:
        idx = np.flatnonzero(df.season == s)
        if not len(idx): continue
        prev = df[df.date < df.date[idx[0]]]
        f = np.bincount(prev.y, minlength=3) / len(prev) if len(prev) else np.array([.46, .25, .29])
        for i in idx:
            out[i] = rps(f, df.y[i])
    return out

PERIODS = {
    "안정기 2011-12~2018-19": lambda df: (df.date >= "2011-08-01") & (df.date < "2019-08-01"),
    "무관중 2020-06-17~2021-05-23": lambda df: (df.date >= "2020-06-17") & (df.date < "2021-08-01"),
    "관중 복귀 2021-22": lambda df: df.season == "2021-22",
    "2022-23~2023-24": lambda df: df.season.isin(["2022-23", "2023-24"]),
    "2024-25": lambda df: df.season == "2024-25",
    "2025-26~": lambda df: df.season.isin(["2025-26", "2026-27"]),
    "전체 2011-12~": lambda df: df.date >= "2011-08-01",
}

CUSUM_SENSITIVE, CUSUM_ROBUST = 14, 16   # 2011-12~2018-19 (무관중 이전)의 잘못된 경보 수로만 정함: 시즌당 약 1번 / 4시즌에 1번

STRATEGIES = {
    "고정 h (현재 실제 코드)":        dict(h_mode="fixed"),
    "장기 기억 (공격만)":             dict(h_mode="long"),
    "항상 짧은 기억 (망각 상시)":     dict(h_mode="short"),
    "장기 + CUSUM 14 (민감)":        dict(h_mode="long", cusum_h=CUSUM_SENSITIVE),
    "장기 + CUSUM 16 (둔감)":        dict(h_mode="long", cusum_h=CUSUM_ROBUST),
    "장기 + 휘슬":                   dict(h_mode="long", whistles=WHISTLES),
    "장기 + 휘슬 + CUSUM 16":        dict(h_mode="long", whistles=WHISTLES, cusum_h=CUSUM_ROBUST),
}

def false_whistle_runs(df, n_false=3, reps=20, seed=0, **kw):
    """무관중 이전 안정기(2011-08~2019-03)의 임의 날짜에 헛휘슬 n_false번. reps번 평균."""
    rng = np.random.default_rng(seed)
    days = df.date[(df.date >= "2011-08-01") & (df.date < "2019-04-01")].unique()
    outs = []
    for _ in range(reps):
        fw = list(rng.choice(days, n_false, replace=False))
        outs.append(run_epl(df, whistles=sorted(fw + list(kw.pop("whistles", WHISTLES))), **kw)[0])
        kw.setdefault("h_mode", "long")
    return np.mean(outs, axis=0)

def block_ci(diff, block=10, reps=2000, seed=0):
    """경기 10개씩 묶은 블록 부트스트랩 95% 구간 (같은 날 경기끼리의 상관 고려)."""
    rng = np.random.default_rng(seed)
    nb = len(diff) // block
    if nb < 2:
        return np.nan, np.nan
    b = diff[: nb * block].reshape(nb, block).mean(1)
    m = rng.choice(b, (reps, nb)).mean(1)
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

def compare(df, losses, base, ref):
    rows = []
    for name, L in losses.items():
        row = {"전략": name}
        for p, f in PERIODS.items():
            msk = f(df).to_numpy()
            row[p] = round((base[msk] - L[msk]).mean() * 1000, 1)
        msk = PERIODS["무관중 2020-06-17~2021-05-23"](df).to_numpy()
        lo, hi = block_ci((losses[ref][msk] - L[msk]) * 1000)
        row["무관중: 장기 기억 대비 (95%)"] = f"{(losses[ref][msk] - L[msk]).mean()*1000:+.1f} [{lo:+.1f}, {hi:+.1f}]"
        msk = PERIODS["전체 2011-12~"](df).to_numpy()
        lo, hi = block_ci((losses[ref][msk] - L[msk]) * 1000)
        row["전체: 장기 기억 대비 (95%)"] = f"{(losses[ref][msk] - L[msk]).mean()*1000:+.1f} [{lo:+.1f}, {hi:+.1f}]"
        rows.append(row)
    return pd.DataFrame(rows).set_index("전략")


In [ ]:

df_epl = load()
base_epl = plan_a(df_epl)
L_epl, alarms_epl, h_epl = {}, {}, {}
for name, kw in STRATEGIES.items():
    L_epl[name], h_epl[name], alarms_epl[name] = run_epl(df_epl, **kw)
L_epl["장기 + 휘슬 + 헛휘슬 3번 (20회 평균)"] = false_whistle_runs(df_epl, 3, 20, h_mode="long", whistles=WHISTLES)
L_epl["장기 + 휘슬 + 헛휘슬 10번 (20회 평균)"] = false_whistle_runs(df_epl, 10, 20, h_mode="long", whistles=WHISTLES)

print(f"경기 수 {len(df_epl):,}  ({df_epl.date.min()} ~ {df_epl.date.max()})")
for name, al in alarms_epl.items():
    if al:
        first = next((x for x in al if x >= "2020-06-17"), None)
        print(f"{name}: 경보 {len(al)}번, 무관중 재개 후 첫 경보 {first}")
compare(df_epl, L_epl, base_epl, ref="장기 기억 (공격만)")


In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(11, 8))
x = pd.to_datetime(df_epl.date)
for name in ["장기 기억 (공격만)", "장기 + 휘슬", "장기 + CUSUM 16 (둔감)", "항상 짧은 기억 (망각 상시)"]:
    axes[0].plot(x, h_epl[name], lw=1, label=name)
for w in WHISTLES:
    axes[0].axvline(pd.to_datetime(w), color="k", ls="--", lw=0.8)
axes[0].set_ylabel("홈 이점 추정 h"); axes[0].set_title("홈 이점 기록: 점선 = 휘슬 (무관중 재개, 관중 복귀)")
axes[0].legend(fontsize=8)

season_gain = pd.DataFrame({n: pd.Series((base_epl - L_epl[n]) * 1000).groupby(df_epl.season).mean()
                            for n in ["장기 기억 (공격만)", "장기 + 휘슬", "장기 + CUSUM 16 (둔감)"]})
season_gain.plot(ax=axes[1], marker="o")
axes[1].axhline(0, color="k", lw=1)
axes[1].set_ylabel("플랜 A 대비 이득 (RPS ×1000)"); axes[1].set_title("총효용 감시: 시즌별 이득")
plt.tight_layout(); plt.show()
season_gain.round(1).T



### 사전 실행 결과

**플랜 A 대비 이득 (RPS ×1000)**과 **장기 기억 대비 차이 (95% 블록 부트스트랩 구간)**

| 전략 | 안정기 2011~2019 | 무관중 | 관중 복귀 2021-22 | 2024-25 | 전체 | 무관중: 장기 대비 | 전체: 장기 대비 |
|---|---|---|---|---|---|---|---|
| 고정 h (현재 실제 코드) | 33.1 | 26.0 | 39.0 | 31.3 | 31.4 | +0.4 [+0.1, +0.7] | +0.1 [−0.1, +0.2] |
| 장기 기억 (공격만) | 33.0 | 25.7 | 39.0 | 31.8 | 31.4 | 기준 | 기준 |
| 항상 짧은 기억 (망각 상시) | 31.2 | 28.6 | 36.0 | 32.5 | 30.1 | +2.9 [−1.9, +7.4] | **−1.3 [−2.1, −0.5]** |
| 장기 + CUSUM 14 (민감) | 32.1 | **29.7** | 36.4 | 32.1 | 30.7 | **+4.0 [+0.3, +7.7]** | −0.7 [−1.3, −0.0] |
| 장기 + CUSUM 16 (둔감) | 32.7 | 28.8 | 36.4 | 32.7 | 31.0 | **+3.1 [+0.2, +5.9]** | −0.3 [−0.9, +0.2] |
| **장기 + 휘슬** | **33.0** | 29.1 | 37.5 | 32.8 | **31.4** | **+3.4 [+0.5, +6.2]** | **+0.1 [−0.3, +0.4]** |
| 장기 + 휘슬 + CUSUM 16 | 32.7 | 28.8 | 36.2 | 32.7 | 31.0 | +3.2 [−0.1, +6.3] | −0.3 [−0.9, +0.2] |
| 장기 + 휘슬 + 헛휘슬 3번 | 32.7 | 29.2 | 37.5 | 32.8 | 31.3 | +3.5 [+0.5, +6.3] | −0.1 [−0.5, +0.3] |
| 장기 + 휘슬 + 헛휘슬 10번 | 32.1 | 29.2 | 37.5 | 32.8 | 31.0 | +3.5 [+0.5, +6.3] | −0.4 [−0.8, +0.0] |

**CUSUM 감지 속도:** 무관중 재개(2020-06-17) 후 첫 경보는 민감 2020-09-14, 둔감 2020-09-20입니다.
재개 후 약 100경기 뒤로, 시뮬레이터에서 플랜 A의 감지 지연을 "약 100건"으로 추정한 것과 같은 수준입니다.

### 시뮬레이터 결론과 대조

| 시뮬레이터 결론 | 실제 데이터 | 판정 |
|---|---|---|
| 진짜 휘슬은 안정 구간 비용 없이 변화 구간 이득을 올린다 | 무관중 +3.4 [+0.5, +6.2], 안정기 손실 0 | **유지** |
| 공격만(장기 기억) 하면 변화에 무너진다 | 무관중 구간 이득이 가장 낮음(25.7). 다만 무너지지는 않음 | **방향은 유지, 크기는 훨씬 작음** |
| CUSUM은 변화를 잡지만 안정 구간에서 보험료를 낸다 | 무관중 +3~4, 안정기 −0.3 ~ −0.9, 전체 약간 손해 | **유지** |
| 감지는 약 100건 늦다 | 첫 경보는 재개 후 약 3개월(약 100경기) 뒤 | **유지** |
| 망각을 항상 켜두면 손해 | 전체 −1.3 [−2.1, −0.5] | **유지 (유의)** |
| 헛휘슬은 비싸다 | 헛휘슬 10번에 전체 −0.4. 시뮬레이터보다 훨씬 쌈 | **방향만 유지**. 기록을 지우지 않고 현재 추정값에서 다시 시작해서 쌉니다 |
| 총효용은 저절로 커지지 않는다 | 시즌별 이득 6.0 → 48.1(2018-19) → 18.7(2025-26), 오르내림 | **유지** |

### 실제 데이터에서 새로 알게 된 것
1. **휘슬에는 방향 정보도 필요합니다.**
   - 관중 복귀 휘슬 뒤 2021-22 시즌 이득이 장기 기억보다 낮았습니다(37.5 vs 39.0).
   - 휘슬이 기록을 "현재 추정값(무관중 시기의 낮은 h)"에서 다시 시작시켰기 때문입니다.
   - 관중 복귀처럼 **이전 상태로 돌아가는 사건**은 "변화 전 기록으로 복귀"하는 휘슬이어야 합니다.
2. **효과 크기가 시뮬레이터보다 훨씬 작습니다.**
   - 홈 이점 채널 관리 방식에 따른 차이는 전체 기준 ±1 이내입니다. 모델 전체 이득(약 31)에 비하면 작습니다.
   - 실제 큰 환경 변화는 16시즌에 한 번(무관중)뿐이었기 때문입니다.
   - 그래서 **이 채널에서는 "공격 위주 + 휘슬"이 가장 좋고, CUSUM은 아직 비용만큼 벌지 못합니다.**
3. **현재 실제 코드(h 고정 0.25)는 장기 기억과 차이가 없습니다(전체 +0.1).** 무관중 같은 사건에 대응하려면 휘슬을 넣는 것이 가장 싼 개선입니다.

### 한계
- 채널이 홈 이점 하나뿐입니다. 팀 실력 기록(감독 교체·이적 같은 팀별 휘슬)은 시험하지 않았습니다.
- 큰 변화가 한 번뿐이라 무관중 구간의 구간 추정이 넓습니다(예: +0.5 ~ +6.2).
- 휘슬 날짜(무관중 재개, 관중 복귀)는 사후에 알고 넣은 것이지만, 당시에도 공개적으로 알려진 사건입니다.
